# Lesson 07 - План дизајн образац

Овај бележник демонстрира **План дизајн образац** за AI агенте користећи Microsoft Agent Framework.
Учићете како да разбили сложене захтеве за путовање на структуриране подзадатке, доделите их специјалистичким агентима,
и извршите резултујући план — све користећи структурирани излаз који покрећу Пидантик модели.


## Подешавање


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Декомпозиција задатка

Декомпозиција задатка је основа планирања у шаблону дизајна. Уместо да тражимо од једног агента да обради сложени захтев од почетка до краја, ми проблем разлажемо на мање, јасно дефинисане **субзадатке**. Сваки субзадатак се додељује специјалистичком агенту (нпр. летови, хотели, активности) са јасним приоритетима и редоследом зависности.

Овај приступ пружа неколико предности:
- **Јасноћа**: сваки субзадатак има једну одговорност.
- **Паралелизам**: независни субзадатци могу да се извршавају паралелно.
- **Поузданост**: неуспеси су изоловани на појединачне субзадатке.
- **Праћење буџета**: трошкови се процењују по субзадатку и збирно приказују.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Креирање планирања агента са структурираним излазом

Планирање агента делује као **координатор на пријему**. С обзиром на захтев за путовање високог нивоа, он производи структурирани `TravelPlan` — разлаже захтев на подзадатке, поставља приоритете и идентификује зависности тако да консијерж или слој за извршење могу обавити посао.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Извођење плана са специјалистичким алатима

Када службеник на рецепцији направи структурирани план, **консијерџ агенти** га извршавају.
Свaki специјалистички алат обрађује једну категорију подзадатака (летови, хотели, активности). Консијерџ
прелази кроз подзадатке плана у редоследу зависности и шаље сваки од њих одговарајућем алату.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Сажетак

У овој лекцији сте научили **Образац за планирање** за AI агенте:

1. **Декомпозиција задатка** — Агент рецепције планира сложен захтев за путовање раздвајајући га у структуриране подзадатке користећи Pydantic моделе, додељујући сваки специјалистичком агенту са приоритетима и зависностима.
2. **Структурирани излаз** — Прослеђивањем `response_format` агент враћа валидирани објекат `TravelPlan` уместо слободног текста, што омогућава поуздану накнадну обраду.
3. **Извођење плана** — Агент консијерж пролази кроз подзадатке користећи специјалистичке алате (`book_flight`, `reserve_hotel`, `book_activity`) да спроведе план и пријави резултате.

Овај образац раздваја *шта радити* (планирање) од *како радити* (извођење), чинећи агенте модуларнијим, тестабилнијим и лакшим за проширење.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ограничење одговорности**:
Овај документ је преведен коришћењем АИ преводилачке услуге [Co-op Translator](https://github.com/Azure/co-op-translator). Иако настојимо да превод буде што тачнији, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати званичним извором информација. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразуме или погрешна тумачења настала коришћењем овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
